# 值迭代算法（Value Iteration）

## 一、导入依赖库

In [6]:
import numpy as np     # 导入 NumPy 库，用于高性能数值计算（多维数组、矩阵运算等）
import random          # 导入 Python 内置 random 模块，可用于生成随机整数（如随机种子）
import importlib.util  # 导入 importlib.util，用于通过文件路径动态加载模块（解决文件名含特殊字符无法直接 import 的问题）

# 文件名 "01.1.Env_GridWorldForVI&PI.py" 含 & 符号和数字开头，不符合 Python 模块标识符规则，
# 因此使用 importlib.util.spec_from_file_location 按路径加载，与 notebook 位于同一目录
_spec = importlib.util.spec_from_file_location(
    "Env_GridWorldForVI_PI",          # 模块名（任意合法 Python 标识符，仅用于内部注册）
    "01.1.Env_GridWorldForVI&PI.py"   # 相对路径；Jupyter 运行时 cwd 默认为 notebook 所在目录
)
GridWorld_v1 = importlib.util.module_from_spec(_spec)  # 创建模块对象；类型：types.ModuleType
_spec.loader.exec_module(GridWorld_v1)                  # 执行模块代码，完成加载（等价于 import）
del _spec                                               # 删除临时变量，保持命名空间整洁

## 二、初始化环境与变量

In [7]:
gamma = 0.9   # 折扣因子（Discount Factor），取值范围 (0,1]；
              # 越接近 0，agent 越"短视"（只关注即时奖励）；
              # 越接近 1，agent 越"远视"（更重视长期累积回报）

rows = 5      # 网格世界的行数，纵轴为 x 方向；需与 desc 列表长度保持一致
columns = 5   # 网格世界的列数，横轴为 y 方向；需与 desc 中每个字符串的长度保持一致

# 实例化 GridWorld 环境，各参数含义：
# - forbiddenAreaScore（float）：进入障碍格（🚫）的即时惩罚奖励，此处为 -10
# - score（float）：到达目标格（✅）的即时正奖励，此处为 1
# - desc（list[str]）：字符串列表描述地图布局；
#   '.' → 普通可通行格，'#' → 障碍格（不可通行），'T' → 目标格
# 返回：GridWorld_v1.GridWorld_v1 对象，封装环境状态、奖励函数和转移规则
gridworld = GridWorld_v1.GridWorld_v1(
    forbiddenAreaScore=-10,
    score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]
)

gridworld.show()  # 打印当前网格地图，⬜️ 普通格，🚫 障碍格，✅ 目标格

# 初始化状态价值函数 V(s)（State-Value Function）
# - 形状：(rows * columns,) = (25,)，一维浮点数组，全零初始化
# - 语义：value[i] 表示从状态 i 出发、遵循最优策略能获得的期望累积折扣回报
# - 可任意初始化（全 0、全随机均可），值迭代算法保证最终收敛到最优价值函数
value = np.zeros(rows * columns)

# 初始化动作价值表 Q(s,a)（Q-table / Action-Value Table）
# - 形状：(rows * columns, 5) = (25, 5)，二维浮点矩阵，全零初始化
# - 语义：qtable[i][j] 表示在状态 i 下执行动作 j 的期望累积折扣回报
# - 5 个动作分别对应：上、右、下、左、原地（具体编号由 GridWorld 定义）
# - 初始值全 0，迭代过程中会被反复覆盖，初始值不影响最终收敛结果
qtable = np.zeros((rows * columns, 5))

# 基于全零 Q 表生成初始贪心策略（Greedy Policy）
# - np.argmax(qtable, axis=1)：对 Q 表每行（每个状态）取动作值最大的索引
# - axis=1 表示沿动作维度（第 1 轴）求 argmax
# - 返回形状：(25,)，dtype=int64；全零 Q 表时所有状态均指向动作 0
policy = np.argmax(qtable, axis=1)

gridworld.showPolicy(policy)  # 用方向箭头可视化当前策略，每格显示该状态的初始动作选择


⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏫️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️


## 三、值迭代算法主循环

> **贝尔曼最优方程（Q 值更新原理）**
>
> 核心公式：$Q(s,a) = r + \gamma \cdot V(s')$
>
> **① 动作价值的定义（展开累积回报）**
>
> $$Q(s,a) = \mathbb{E}\!\left[\, r_0 + \gamma r_1 + \gamma^2 r_2 + \cdots \mid s_0=s,\; a_0=a \,\right]$$
>
> 在状态 $s$ 强制执行动作 $a$，之后遵循策略 $\pi$ 走到底的**期望总折扣回报**。
>
> **② 拆分即时奖励与未来回报**
>
> $$Q(s,a) = r + \gamma \cdot \mathbb{E}[V(s')]$$
>
> 括号内 $r_1 + \gamma r_2 + \cdots$ 正是从下一状态 $s'$ 出发的累积回报，即状态价值 $V(s')$。
>
> **③ 确定性环境下消去期望**
>
> 本环境中执行动作后 $s'$ 唯一确定（无随机性），故 $\mathbb{E}[V(s')] = V(s')$，最终得：
>
> $$\boxed{Q(s,a) = r + \gamma \cdot V(s')}$$
>
> **注意**：即时奖励 $r$（`score`）$\neq$ 动作价值 $Q(s,a)$；$r$ 只是"这一步当场拿到的奖励"，是计算 $Q$ 的原材料之一；$Q(s,a)$ 才是完整的动作价值（当前奖励 + 所有未来折扣回报之和）。

> **状态价值与动作价值的关系**
>
> **一般情况（随机策略 $\pi$）：**
>
> $$V^\pi(s) = \sum_a \pi(a \mid s) \cdot Q^\pi(s,a)$$
>
> 对所有动作的价值按策略概率**加权求期望**（"把动作平均掉"），得到状态价值。
>
> **值迭代中（确定性贪心策略）：**
>
> 每个状态只选 Q 值最大的动作（该动作概率为 1，其余为 0），加权平均退化为取最大值：
>
> $$\boxed{V^*(s) = \max_a\, Q(s, a)}$$
>
> **值迭代的更新闭环：**
>
> 旧 $V(s)$ $\xrightarrow{Q(s,a)=r+\gamma V(s')}$ $Q(s,a)$ $\xrightarrow{\max_a}$ 新 $V(s)$ $\xrightarrow{\text{反复迭代}}$ $V^*(s)$
>
> 每一轮先用**旧状态价值**算出**动作价值**，再用**动作价值**更新**状态价值**，循环直至收敛。

In [8]:
# 初始化"前一轮"价值向量，令其与 value 不相等，确保第一次迭代时能进入 while 循环
# - value.copy()：深拷贝 value（当前全零数组），避免引用同一对象
# - +1：使 pre_value 与 value 数值不同，保证收敛判断条件首次为真
# - 形状：(25,)，dtype=float64
pre_value = value.copy() + 1

gridworld.show()              # 打印网格世界地图，确认当前环境配置
gridworld.showPolicy(policy)  # 打印初始策略（当前全部指向动作 0 方向的箭头）

cnt = 0  # 外层迭代计数器，用于安全限制最大迭代轮次（理论上值迭代必然收敛，此计数器仅作安全阀）

# ── 外层收敛循环 ─────────────────────────────────────────────────────────────
# 收敛判断：np.sum((pre_value - value) ** 2) 计算相邻两轮价值向量的 L2 距离的平方
# 当变化量 ≤ 0.001 时认为价值函数已收敛，退出循环
SEP = "=" * 50  # 分隔线字符串，用于在每轮迭代输出之间添加视觉分隔，宽度 50 个字符

while np.sum((pre_value - value) ** 2) > 0.001:

    pre_value = value.copy()  # 保存当前价值向量的深拷贝，供下一轮计算收敛误差使用

    cnt = cnt + 1   # 迭代轮次加 1
    if cnt > 100:   # 安全阀：超过 100 轮强制退出，防止因代码 bug 导致无限循环
        break

    # ── 内层循环：遍历所有状态，更新 Q(s,a) ─────────────────────────────────
    for i in range(rows * columns):  # i：状态编号，范围 [0, 24]，共 rows×columns=25 个状态

        nowx = i // columns  # 将一维状态编号 i 转换为网格行坐标（整除列数得行号），当前暂未直接使用
        nowy = i % columns   # 将一维状态编号 i 转换为网格列坐标（取模列数得列号），当前暂未直接使用

        for j in range(5):  # j：动作编号，共 5 个动作（上/右/下/左/原地，由 GridWorld 定义）

            # 调用环境接口获取状态转移信息
            # getScore(state: int, action: int) → (reward: float, next_state: int)
            # - score（float）：在状态 i 执行动作 j 后获得的即时奖励 r
            # - nextState（int）：执行动作后转移到的下一状态编号
            score, nextState = gridworld.getScore(i, j)

            # 贝尔曼最优方程 Q(s,a) = r + γ·V(s')，推导见三.1节
            # score→r（即时奖励），gamma→γ（折扣因子），value[nextState]→V(s')，qtable[i][j]→Q(s,a)
            qtable[i][j] = score + gamma * value[nextState]

    # ── 策略提升（Policy Improvement）────────────────────────────────────────
    # 对每个状态选取 Q 值最大的动作作为新策略（确定性贪心策略）
    # np.argmax(qtable, axis=1)：沿动作维度（axis=1）取最大 Q 值对应的动作索引
    # 返回形状：(25,)，dtype=int64
    # 值迭代中策略百分百选 argmax 动作（概率为 1），无须处理随机策略
    policy = np.argmax(qtable, axis=1)

    # 值迭代贪心策略下 V*(s) = max_a Q(s,a)，状态价值由动作价值取最大得到（详见三.2节）
    # np.max(qtable, axis=1)：沿动作维度（axis=1）取最大 Q 值；返回形状 (25,)，dtype=float64
    value = np.max(qtable, axis=1)

    # ── 本轮输出：分隔线 + 标题行 + 价值矩阵 + 策略可视化 ─────────────────────
    print(SEP)  # 打印分隔线，与上一轮输出形成视觉边界
    # 打印本轮标题：轮次编号 + 收敛距离（L2 距离平方），便于追踪迭代进度
    print(f"第 {cnt:>2d} 轮迭代 | 收敛距离（Δ²）: {np.sum((pre_value - value) ** 2):.4f}")
    print("── 状态价值函数 V(s) ──")  # 小节标题，说明下方矩阵含义
    # 将一维价值向量 reshape 为 (rows, columns) = (5, 5) 的矩阵后保留 1 位小数打印
    print(np.round(value.reshape(rows, columns), 1))
    print("── 当前策略 π(s) ──")  # 小节标题，说明下方策略箭头含义
    gridworld.showPolicy(policy)  # 打印本轮策略提升后的新策略可视化结果

# ── 收敛后输出最终结果 ────────────────────────────────────────────────────────
print(SEP)  # 最终分隔线，与迭代过程输出形成明显边界
print(f"收敛完成，共迭代 {cnt} 轮")  # 打印总迭代轮数
print("── 最终状态价值函数 V*(s) ──")  # 标题，说明下方为最优价值函数
# 打印最终收敛的最优价值函数矩阵，形状：(rows, columns) = (5, 5)，保留 1 位小数
print(np.round(value.reshape(rows, columns), 1))
print("── 最终最优策略 π*(s) ──")  # 标题，说明下方为最优策略向量
print(policy)  # 打印最终最优策略向量，形状：(25,)，dtype=int64，每个元素为对应状态的最优动作编号

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏫️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️
第  1 轮迭代 | 收敛距离（Δ²）: 5.0000
── 状态价值函数 V(s) ──
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 1. 1. 0.]
 [0. 0. 1. 0. 0.]]
── 当前策略 π(s) ──
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️➡️⬆️
第  2 轮迭代 | 收敛距离（Δ²）: 5.6700
── 状态价值函数 V(s) ──
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  1.9 0.  0. ]
 [0.  1.9 1.9 1.9 0. ]
 [0.  0.9 1.9 0.9 0. ]]
── 当前策略 π(s) ──
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬆️
第  3 轮迭代 | 收敛距离（Δ²）: 5.2488
── 状态价值函数 V(s) ──
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  2.7 0.  0. ]
 [0.  2.7 2.7 2.7 0. ]
 [0.  1.7 2.7 1.7 0.8]]
── 当前策略 π(s) ──
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬅️
第  4 轮迭代 | 收敛距离（Δ²）: 4.7830
── 状态价值函数 V(s) ──
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  3.4 0.  0. ]
 [0.  3.4 3.4 3.4 0.7]
 [0.  2.4 3.4 2.4 1.5]]
── 当前策略 π(s) ──
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
第  

## 四、辅助功能测试

In [9]:
print('⬜️🚫')              # 打印普通格（⬜️）与障碍格（🚫）的 Emoji，验证当前环境能否正常渲染
print('⬜️✅')              # 打印普通格与目标格（✅）的 Emoji
print('⬆️➡️⬇️⬅️🔄')       # 打印四个基本方向箭头（上/右/下/左）及原地（🔄）Emoji
print('⏫︎⏩️⏪🔄✅')        # 打印快进/快退方向 Emoji（⏫/⏩/⏪），验证双倍速箭头的渲染效果

tmp = "⏫︎⏩️⏬⏪🔄"  # 将多个 Emoji 字符拼接为字符串，用于测试 Python 字符串的 Unicode 码点索引行为
# 注意：部分 Emoji（如 ⏫︎）由多个 Unicode 码点组成（基础符号 + 变体选择符 U+FE0E/U+FE0F）；
# Python 的 str 下标按 Unicode 码点（code point）索引，而非视觉上的"字形"数量，
# 因此索引结果可能与直观预期不符（例如索引到不可见的变体选择符）

print(tmp[0])  # 打印 tmp 中索引 0 处的 Unicode 码点字符（⏫ 的基础字符，U+23EB）
print(tmp[1])  # 打印 tmp 中索引 1 处的字符（可能是不可见的文字变体选择符 U+FE0E，显示为空行）
print(tmp[2])  # 打印 tmp 中索引 2 处的字符（⏩ 的基础字符，U+23E9）

⬜️🚫
⬜️✅
⬆️➡️⬇️⬅️🔄
⏫︎⏩️⏪🔄✅
⏫
︎
⏩
